# MCP 구조와 capability 협상

MCP 클라이언트와 서버는 도구·리소스·프롬프트를 어떻게 구분하고 서로 지원하는 기능을 어떻게 확인할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. 비밀 값은 출력하지 않습니다.

## 패키지 버전 확인

패키지 버전 확인에 필요한 값과 결과를 확인합니다.

In [1]:
# 패키지 버전 확인에 필요한 값을 준비하고 결과를 확인합니다.
%pip install -q "langchain[mcp]==1.4.0" langchain-openai==1.6.0 fastmcp==4.0.3 mcp==2.2.0 python-dotenv==1.2.3
import warnings
from importlib.metadata import version
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", message=r".*langchain\.mcp.*", category=LangChainBetaWarning)

for package in ["langchain", "langchain-openai", "fastmcp", "mcp"]:
    print(package, version(package))

Note: you may need to restart the kernel to use updated packages.


langchain 1.4.0
langchain-openai 1.6.0
fastmcp 4.0.3
mcp 2.2.0


## 학생용 서버 불러오기

학생용 서버 불러오기에 필요한 값과 결과를 확인합니다.

In [2]:
# 학생용 서버 불러오기에 필요한 값을 준비하고 결과를 확인합니다.
import importlib.util
from pathlib import Path

relative = Path("labs/part6/ch28-mcp/course_server.py")
server_path = next(folder / relative for folder in [Path.cwd(), *Path.cwd().parents] if (folder / relative).is_file())
assert server_path.is_file(), f"서버 파일을 찾지 못했습니다: {server_path}"
spec = importlib.util.spec_from_file_location("course_server", server_path)
course_server = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(course_server)
print("server:", course_server.mcp.name)

server: 교육센터 자료실


## 세 capability 발견

세 capability 발견에 필요한 값과 결과를 확인합니다.

In [3]:
# 세 capability 발견에 필요한 값을 준비하고 결과를 확인합니다.
import asyncio
from fastmcp import Client

async def discover() -> tuple[list, list, list]:
    async with Client(course_server.mcp) as client:
        tools = await client.list_tools()
        resources = await client.list_resources()
        prompts = await client.list_prompts()
        return tools, resources, prompts

tools, resources, prompts = await discover()
print("tools:", [item.name for item in tools])
print("resources:", [str(item.uri) for item in resources])
print("prompts:", [item.name for item in prompts])
assert (len(tools), len(resources), len(prompts)) == (1, 1, 1)

tools: ['lookup_policy']
resources: ['policy://handbook']
prompts: ['explain_policy']


## 도구·리소스·프롬프트 실제 사용

도구·리소스·프롬프트 실제 사용에 필요한 값과 결과를 확인합니다.

In [4]:
# 도구·리소스·프롬프트 실제 사용에 필요한 값을 준비하고 결과를 확인합니다.
async def use_components() -> tuple[object, list, object]:
    async with Client(course_server.mcp) as client:
        tool_result = await client.call_tool("lookup_policy", {"topic": "대출"})
        resource_result = await client.read_resource("policy://handbook")
        prompt_result = await client.get_prompt("explain_policy", {"topic": "대출"})
        return tool_result, resource_result, prompt_result

tool_result, resource_result, prompt_result = await use_components()
print("tool:", tool_result.content[0].text)
print("resource:", resource_result[0].text)
print("prompt:", prompt_result.messages[0].content.text)
assert "14일" in tool_result.content[0].text
assert "운영:" in resource_result[0].text
assert "대출" in prompt_result.messages[0].content.text

tool: 일반 도서는 1인 5권까지 14일 동안 대출할 수 있습니다.
resource: 대출: 일반 도서는 1인 5권까지 14일 동안 대출할 수 있습니다.
운영: 평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.
연체: 연체한 일수만큼 새 대출이 제한됩니다.
prompt: 대출 규정을 근거 문장과 함께 두 문장으로 설명해 주세요.
